In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime
 
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
sklepy = ['Warszawa', 'Krakow', 'Gdansk', 'Wroclaw']
kategorie = ['elektronika', 'odziez', 'żywnosc', 'ksiazki']
 
def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }
 
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.3)
 
producer.flush()
producer.close()

Overwriting producer.py


In [2]:
import pandas as pd
import numpy as np
 
np.random.seed(42)
 
# Generujemy TYLKO normalne transakcje — fraudów nie potrzebujemy!
N_NORMAL = 2000
 
normal = pd.DataFrame({
    'amount':        np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute':  np.random.poisson(3, N_NORMAL),
})
 
print(f"Dane treningowe: {len(normal)} normalnych transakcji")
print()
normal.describe().round(2)

Dane treningowe: 2000 normalnych transakcji



,amount,is_electronics,tx_per_minute
count,2000.00,2000.00,2000.00
mean,253.77,0.28,2.96
std,324.84,0.45,1.65
min,5.81,0.00,0.00
25%,79.63,0.00,2.00
50%,155.20,0.00,3.00
75%,293.82,1.00,4.00
max,5000.00,1.00,10.00


In [3]:
from sklearn.ensemble import IsolationForest
import pickle
 
features = ['amount', 'is_electronics', 'tx_per_minute']
X_train = normal[features]
 
# contamination: spodziewamy się ~5% anomalii w strumieniu
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)
iso_forest.fit(X_train)
 
print("Model wytrenowany.")
print(f"Liczba drzew: {iso_forest.n_estimators}")
print(f"Contamination: {iso_forest.contamination}")
 
# Zapisz model
with open('fraud_model_if.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("\nZapisano do fraud_model_if.pkl")

Model wytrenowany.
Liczba drzew: 100
Contamination: 0.05

Zapisano do fraud_model_if.pkl


In [4]:
test_cases = pd.DataFrame([
    {'amount': 150.0,  'is_electronics': 0, 'tx_per_minute': 3,  'opis': 'normalna'},
    {'amount': 89.0,   'is_electronics': 0, 'tx_per_minute': 2,  'opis': 'normalna'},
    {'amount': 4800.0, 'is_electronics': 1, 'tx_per_minute': 12, 'opis': 'podejrzana'},
    {'amount': 3500.0, 'is_electronics': 1, 'tx_per_minute': 9,  'opis': 'podejrzana'},
    {'amount': 250.0,  'is_electronics': 1, 'tx_per_minute': 5,  'opis': 'graniczna'},
])

 

X_test = test_cases[features]
preds  = iso_forest.predict(X_test)          # +1 lub -1
scores = iso_forest.decision_function(X_test) # anomaly score

 

test_cases['wynik']  = preds
test_cases['score']  = scores.round(4)
test_cases['anomalia'] = test_cases['wynik'] == -1

 

print(test_cases[['opis', 'amount', 'is_electronics', 'tx_per_minute',
                   'wynik', 'score', 'anomalia']].to_string(index=False))



      opis  amount  is_electronics  tx_per_minute  wynik   score  anomalia
  normalna   150.0               0              3      1  0.2115     False
  normalna    89.0               0              2      1  0.2097     False
podejrzana  4800.0               1             12     -1 -0.1932      True
podejrzana  3500.0               1              9     -1 -0.1894      True
 graniczna   250.0               1              5      1  0.0641     False


In [5]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np
 
app = FastAPI(title="Fraud Detection API — Isolation Forest")
 
model = pickle.load(open('fraud_model_if.pkl', 'rb'))
 
class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int
 
@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction     = model.predict(X)[0]           # +1 lub -1
    anomaly_score  = model.decision_function(X)[0]  # ujemny = bardziej podejrzany
 
    # Normalizujemy score do przedziału [0, 1] — dla spójności z Ćw. 2
    # decision_function typowo zwraca wartości z zakresu [-0.5, 0.5]
    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))
 
    return {
        "is_fraud":          bool(prediction == -1),
        "fraud_probability": round(fraud_probability, 4),
        "model":             "isolation_forest",
    }
 
@app.get("/health")
def health():
    return {"status": "ok"}

Writing fraud_api.py


In [7]:
import requests 
r= requests.get("http://localhost:8001/health")
r.json()

{'status': 'ok'}

In [1]:
import requests, time

 

time.sleep(1)  # daj chwilę na restart serwera

 

cases = [
    {"amount": 150,  "is_electronics": 0, "tx_per_minute": 3,  "opis": "normalna"},
    {"amount": 4800, "is_electronics": 1, "tx_per_minute": 12, "opis": "podejrzana"},
    {"amount": 89,   "is_electronics": 0, "tx_per_minute": 2,  "opis": "normalna"},
    {"amount": 3200, "is_electronics": 1, "tx_per_minute": 8,  "opis": "podejrzana"},
]

 

for case in cases:
    payload = {k: v for k, v in case.items() if k != 'opis'}
    r = requests.post("http://localhost:8001/score", json=payload)
    result = r.json()
    print(f"[{case['opis']:10s}] amount={case['amount']:5} "
          f"→ fraud={result['is_fraud']}, prob={result['fraud_probability']:.3f}")



[normalna  ] amount=  150 → fraud=False, prob=0.288
[podejrzana] amount= 4800 → fraud=True, prob=0.693
[normalna  ] amount=   89 → fraud=False, prob=0.290
[podejrzana] amount= 3200 → fraud=True, prob=0.678


In [2]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests
BROKER = "broker:9092"
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers=BROKER,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)
alert_producer = KafkaProducer(
    bootstrap_servers=BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

Writing ml_consumer.py


In [3]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests
BROKER = "broker:9092"
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers=BROKER,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)
alert_producer = KafkaProducer(
    bootstrap_servers=BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
API_URL = "http://localhost:8001/score"
for message in consumer:
    tx = message.value
    is_electronics = 1 if tx.get('category') == 'elektronika' else 0
    features = {
        "amount": tx['amount'],
        "is_electronics": is_electronics,
        "tx_per_minute": 5
    }
    try:
        response = requests.post(API_URL, json=features, timeout=2)
        result = response.json()
    except requests.RequestException as e:
        print(f"API niedziala: {e} ")
        continue
    if result['is_fraud']:
        alert = {
            **tx,
            'fraud_probability': result['fraud_probability'],
            'alert_source': 'ml_model'
        }
        alert_producer.send('alerts', value=alert)
        print(
            f"FRAUD {result['fraud_probability']}, {tx['tx_id']}")
alert_producer.flush()


Overwriting ml_consumer.py
